In [ ]:
# حل التعارض: datasets==3.6.0 بتحتاج fsspec[http]==2023.6.0
!pip -q install -U "fsspec[http]==2023.6.0" "datasets==3.6.0"

# ثبّت الباقي بدون كسر الاعتمادات
!pip -q install -U "transformers>=4.43,<5" "accelerate>=0.33,<1" \
                   "sentence-transformers>=2.5,<3" "faiss-cpu>=1.7,<2" \
                   "pypdf>=3,<5" "safetensors>=0.4.2" "huggingface_hub>=0.23,<1"

# تأكيد الإصدارات (اختياري)
import importlib, pkgutil
for m in ["fsspec","datasets","transformers","accelerate","sentence_transformers","faiss","pypdf"]:
    try:
        mod = importlib.import_module(m if m!="faiss" else "faiss")
        print(m, getattr(mod, "__version__", ""))
    except Exception as e:
        print(m, "ERR:", e)


In [ ]:
# If needed (Kaggle/Colab): uncomment & run
%pip install -q  torch accelerate bitsandbytes fastapi uvicorn nest_asyncio pyngrok requests

In [ ]:
# لو الريبو Private
from huggingface_hub import login
login()  # هيفتح لك إدخال التوكن



In [ ]:
pip install -U bitsandbytes

In [ ]:
import torch, os, re
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# UPGRADED to Llama 3.1 8B
REPO_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tok = AutoTokenizer.from_pretrained(REPO_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    REPO_ID,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loaded:", REPO_ID, "| 4-bit mode active")


In [ ]:
import os
if "vectordb" not in globals():
    from pypdf import PdfReader
    from sentence_transformers import SentenceTransformer
    import faiss
    import numpy as np

    PDF_PATH = "/content/Egiptura_Database_RAG_Fixed.pdf"  # عدّل المسار لو لزم

    def load_pdf_text(path):
        reader = PdfReader(path)
        pages = [p.extract_text() or "" for p in reader.pages]
        text = "\n".join(pages)
        return text

    def chunk_text(text, chunk_size=900, overlap=150):
        chunks = []
        i = 0
        while i < len(text):
            chunk = text[i:i+chunk_size]
            chunks.append(chunk)
            i += (chunk_size - overlap)
        return [c.strip() for c in chunks if c.strip()]

    class SimpleVectorDB:
        def __init__(self, texts, embedder_name="sentence-transformers/all-MiniLM-L6-v2"):
            self.model = SentenceTransformer(embedder_name)
            self.texts = texts
            emb = self.model.encode(texts, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
            self.index = faiss.IndexFlatIP(emb.shape[1])
            self.index.add(emb.astype(np.float32))

        def similarity_search(self, query, k=2):
            q = self.model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
            D, I = self.index.search(q, k)
            class Doc:  # بسيط
                def __init__(self, page_content): self.page_content = page_content
            return [Doc(self.texts[i]) for i in I[0] if i >= 0]

    print("Building VectorDB from:", PDF_PATH)
    pdf_text = load_pdf_text(PDF_PATH)
    chunks = chunk_text(pdf_text)
    vectordb = SimpleVectorDB(chunks)
    print("VectorDB ready. Chunks:", len(chunks))
else:
    print("Using existing vectordb.")


In [ ]:
# ===== Cell 1 — Setup & deps (UPDATED) =====
!pip -q install langchain-community langchain-text-splitters faiss-cpu pypdf sentence-transformers rank_bm25

import os, re, glob, torch, warnings, pickle
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


In [ ]:
# ===== Cell 2 — Hybrid Index (FAISS + BM25) =====
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from rank_bm25 import BM25Okapi
import pickle, glob, os

class E5Embeddings(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        return super().embed_documents([f"passage: {t}" for t in texts])
    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EMB_MODEL = "intfloat/multilingual-e5-base"
emb = E5Embeddings(
    model_name=EMB_MODEL,
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)

INDEX_DIR = "egiptura_hybrid_index"
BM25_INDEX = None
DOCS_CHUNKS = []

def build_hybrid_index(pdf_paths, persist_dir=INDEX_DIR):
    if not pdf_paths:
        raise RuntimeError("No PDF files found to index. Please upload Egiptura_Database_RAG_Fixed.pdf")

    docs = []
    for p in pdf_paths:
        try:
            print(f"[i] Attempting to load: {p}")
            loader = PyPDFLoader(p)
            pages = loader.load()
            for i, d in enumerate(pages, start=1):
                d.metadata.update({"source": os.path.basename(p), "page": i})
            docs.extend(pages)
        except Exception as e: print(f"[!] Skip {p}: {e}")

    if not docs:
        raise RuntimeError("PDF files were found but no text could be extracted.")

    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
    chunks = splitter.split_documents(docs)

    if not chunks:
        raise RuntimeError("Extracted text resulted in zero chunks.")

    print(f"[i] Building FAISS for {len(chunks)} chunks...")
    vdb = FAISS.from_documents(chunks, emb)
    vdb.save_local(persist_dir)

    print("[i] Building BM25 index...")
    texts = [c.page_content for c in chunks]
    tokenized = [t.lower().split() for t in texts]
    bm25 = BM25Okapi(tokenized)
    with open(os.path.join(persist_dir, "bm25.pkl"), "wb") as f:
        pickle.dump({"bm25": bm25, "chunks": chunks}, f)

    return vdb, bm25, chunks

def load_hybrid_index(persist_dir=INDEX_DIR):
    faiss_path = os.path.join(persist_dir, "index.faiss")
    bm25_path = os.path.join(persist_dir, "bm25.pkl")

    if os.path.exists(faiss_path) and os.path.exists(bm25_path):
        print("[i] Loading existing hybrid index...")
        vdb = FAISS.load_local(persist_dir, emb, allow_dangerous_deserialization=True)
        with open(bm25_path, "rb") as f:
            data = pickle.load(f)
        return vdb, data["bm25"], data["chunks"]

    print("[i] Index not found or incomplete. Searching for PDFs...")
    # More robust path search for both Kaggle and Colab
    patterns = [
        "*.pdf",
        "**/*.pdf",
        "/kaggle/input/**/*.pdf",
        "/content/**/*.pdf"
    ]
    paths = []
    for pat in patterns:
        paths.extend(glob.glob(pat, recursive=True))

    # Remove duplicates and prioritize likely database files
    paths = sorted(list(set(paths)), key=lambda x: "Egiptura" in x, reverse=True)

    if not paths:
        print("[!] Current directory files:", os.listdir('.'))

    return build_hybrid_index(paths)

vectordb, BM25_INDEX, DOCS_CHUNKS = load_hybrid_index()

try:
    from sentence_transformers import CrossEncoder
    RERANKER = CrossEncoder("BAAI/bge-reranker-v2-m3", device=DEVICE)
    print("[✓] Reranker ready.")
except Exception as e: RERANKER = None


In [ ]:
# ===== Cell 1 — Config + Regex + Utilities (RESTORED) =====
import re, time, torch

_MAX_CTX   = 250
_MAX_NEW   = 96
_NO_REPEAT = 4
_REP_PEN   = 1.1

_AR  = re.compile(r"[\u0600-\u06FF]")
_LAT = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]")

_SPECIAL         = re.compile(r"(?:<\|[^>|]+?\|>)|(?:</?s>)|(?:<<SYS>>|<</SYS>>)|(?:\[\s*/?\\s*INST\s*\])|(?:<!\[CDATA\[|\]\]>)|(?:<\/?\w+\s*>)", re.IGNORECASE)
_PREFIX_LINE     = re.compile(r'(?im)^\s*(?:A|Ans|Resp|Rpta|Respuesta|Answer|الإجابة)\s*:?\s*')
_PREFIX_INLINE   = re.compile(r'(?i)\b(?:respuesta|answer|الإجابة)\s*:?\s*')
_DISCLAIMER      = re.compile(r'^\s*(?:nota|note|ملاحظة|تنبيه)\s*[:\-–]', re.IGNORECASE)
_EMPTY_LIST_LINE = re.compile(r'^\s*(?:[-–—•\u2022]|\(?\d{1,3}\)?[.)]?)\s*$')
_DECOR_LINE      = re.compile(r'^[.\-=_~•·\u2022\s]{4,}$')
_EMOJI_RE        = re.compile(r"[\U0001F300-\U0001F6FF\U0001F900-\U0001FAFF\U0001F1E6-\U0001F1FF\u2600-\u26FF\u2700-\u27BF]")

_JUNK_TOK_RE     = re.compile(r"(?:</?s>|<s>|</s>|<<SYS>>|<</SYS>>|\[/?INST\])", re.I)
_JUNK_SYMBOLS_RE = re.compile(r"[<>\[\]\|/\\]{1,}")
_TAGGY_RE        = re.compile(r"</?[A-Za-z][^>]{0,20}>")

_PRICE = re.compile(
    r"(?:\$\s?\d+(?:[,\.\s]\d+)*)|"
    r"(?:\d+\s?(?:USD|EUR|EGP|AED|SAR))|"
    r"(?:(?:USD|EUR|EGP|AED|SAR)\s?\d+(?:[,\.\s]\d+)*)|"
    r"(?:\d+\s?(?:دولار|يورو|جنيه|ريال|درهم))|"
    r"(?:precio|coste|costo|price)\s*(?:aprox\.?|estimado)?\s*:?[\s~]*\d+",
    re.IGNORECASE
)

_QLINE_RE = re.compile(r"\s*(?:[¿].*|\S.*[?؟])\s*$")
_AR_CH    = r"[\u0600-\u06FF]"
_LAT_CH   = r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]"

def _detect_lang(q: str) -> str:
    s = q.strip().lower()
    if _AR.search(s):
        return "ar"
    if re.search(r"[áéíóúüñ¿¡]", s) or any(w in s for w in (
        "hola","buenas","gracias","quiero","viaje","programa","itinerario",
        "semana","egipto","viajar","turismo","precio","coste","costo",
        "cuanto","cuánto"
    )):
        return "es"
    return "en"

def _wants_price(q: str) -> bool:
    s = q.lower()
    return any(k in s for k in [
        "price","cost","budget","how much","cuanto","cuánto","precio","coste","costo","tarifa",
        "سعر","التكلفة","تكلفة","ميزانية","كم تكلف","كم ممكن تكلف","السعر"
    ])

def _is_factoid(q: str) -> bool:
    ql = q.strip().lower()
    words = len(ql.split())
    return words <= 12 and bool(re.search(
        r"\b(quien|quién|que|qué|cuando|cuándo|donde|dónde|who|what|when|where|من|ما|متى|أين)\b", ql))

def _trim_tokens(text: str, max_tokens: int) -> str:
    global tok, tokenizer
    tok_ = tok if 'tok' in globals() else tokenizer
    ids = tok_.encode(text, add_special_tokens=False)
    return text if len(ids) <= max_tokens else tok_.decode(ids[:max_tokens], skip_special_tokens=True)

def _strip_decor(text: str) -> str:
    out = []
    for ln in text.splitlines():
        if _DECOR_LINE.match(ln.strip()):
            continue
        ln = re.sub(r'([.\-=_~•·\u2022])\1{2,}', r'\1\1', ln)
        out.append(ln)
    return "\n".join(out).strip()

def _strip_emojis(text: str) -> str:
    return _EMOJI_RE.sub("", text)

def _strip_token_garbage(t: str) -> str:
    t = _JUNK_TOK_RE.sub("", t)
    t = _TAGGY_RE.sub("", t)
    t = _JUNK_SYMBOLS_RE.sub(" ", t)
    t = re.sub(r"\s{2,}", " ", t)
    return t.strip()

def _strip_system_prompt(text: str) -> str:
    """Removes trailing prompt leakage artifacts."""
    patterns = [
        r"(?i)You are Egiptura's assistant.*",
        r"(?i)Reply ONLY in English.*",
        r"(?i)Use ONLY the provided context.*",
        r"(?i)end\. s\. s\..*",
        r"(?i)Was that correct\? If so.*",
        r"(?i)Answered by.*",
        r"(?i)Follow us on.*",
        r"(?i)Contact us via.*",
        r"(?i)We look forward.*",
        r"(?i)How can I help you further.*",
        r"(?i)Please provide your contact.*",
    ]
    for p in patterns:
        text = re.split(p, text)[0]
    return text.strip()

def _clean(text: str) -> str:
    text = _strip_system_prompt(text)
    text = _SPECIAL.sub("", text)
    text = _PREFIX_LINE.sub("", text)
    text = _PREFIX_INLINE.sub("", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    text = "\n".join([ln for ln in text.splitlines() if not _DISCLAIMER.match(ln.strip())])
    text = "\n".join([ln for ln in text.splitlines() if not _EMPTY_LIST_LINE.match(ln)])
    lines, seen = [], set()
    for ln in text.splitlines():
        k = ln.strip()
        if not k:
            lines.append("")
            continue
        if k not in seen:
            seen.add(k); lines.append(ln)
    text = "\n".join(lines).strip()
    text = _strip_decor(text)
    return text

def _enforce_lang(text: str, lang: str) -> str:
    def keep_line(line: str) -> bool:
        ar  = sum(1 for ch in line if _AR.match(ch))
        lat = sum(1 for ch in line if re.match(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", ch))
        if ar == 0 and lat == 0:
            return True
        if lang == "ar": return ar  >= max(2, int(1.2 * lat))
        if lang == "es": return lat >= max(2, int(1.0 * ar))
        return lat >= max(2, int(1.0 * ar))
    return "\n".join([ln for ln in text.splitlines() if keep_line(ln)])

_PARENS_RE = re.compile(r"\(([^)]{1,160})\)")
_BRACK_RE  = re.compile(r"\[([^\]]{1,160})\]")

def _strip_cross_parentheticals(text: str, lang: str) -> str:
    def bad_chunk(chunk: str) -> bool:
        has_ar  = bool(_AR.search(chunk))
        has_lat = bool(re.search(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]", chunk))
        if lang == "ar":  return has_lat and not has_ar
        if lang == "es":  return has_ar and not has_lat
        return has_ar
    def repl_parens(m): return "" if bad_chunk(m.group(1)) else m.group(0)
    def repl_brack(m):  return "" if bad_chunk(m.group(1)) else m.group(0)
    text = _PARENS_RE.sub(repl_parens, text)
    text = _BRACK_RE.sub(repl_brack, text)
    return text

def _strip_prices_if_needed(text: str, allow: bool) -> str:
    return text if allow else _PRICE.sub("▢", text).strip()

def _is_greeting(q: str) -> bool:
    return bool(re.search(r"\b(hola|buenas|hi|hello|hey|salut|مرحبا|اهلا|أهلاً|السلام عليكم)\b", q.strip().lower()))

def _normalize_question(q: str, lang: str) -> str:
    q = _strip_emojis(q)
    q = re.sub(r"[?؟]{2,}", "?", q)
    q = re.sub(r"[!¡]{2,}", "!", q)
    q = re.sub(r"[.\u2026]{3,}", "…", q)
    q = re.sub(r"\s{2,}", " ", q).strip()
    if lang == "ar":
        q = q.replace("¿", "").replace("?", "؟")
        if not q.endswith("؟"): q = q.rstrip("؟") + "؟"
    elif lang == "es":
        q = q.rstrip("?") + "?"
        if not q.startswith("¿"): q = "¿" + q
    else:
        q = q.replace("¿", "")
        q = q.rstrip("?") + "?"
    return q.strip()

def _line_matches_lang(line: str, lang: str) -> bool:
    has_ar  = bool(re.search(_AR_CH, line))
    has_lat = bool(re.search(_LAT_CH, line))
    if lang == "ar": return has_ar and not (has_lat and not has_ar)
    if lang == "es": return has_lat and not has_ar
    return has_lat and not has_ar

def _enforce_one_followup(text: str, lang: str) -> str:
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    keep_idx = None
    for i in range(len(lines) - 1, -1, -1):
        ln = lines[i]
        if _QLINE_RE.match(ln) and _line_matches_lang(ln, lang):
            keep_idx = i
            break
    cleaned = []
    for i, ln in enumerate(lines):
        if _QLINE_RE.match(ln):
            if i == keep_idx:
                cleaned.append(ln)
        else:
            cleaned.append(ln)
    if keep_idx is None:
        fallback = {
            "ar": "هل تود أن أضيف تفاصيل محددة؟",
            "es": "¿Quieres que añada algún detalle en concreto?",
            "en": "Would you like me to add any specific details?",
        }.get(lang, "Would you like me to add any specific details?")
        cleaned.append(fallback)
    else:
        if cleaned and _QLINE_RE.match(cleaned[-1]):
            cleaned[-1] = _normalize_question(cleaned[-1], lang)
    cleaned = [_strip_emojis(ln) for ln in cleaned]
    return "\n".join([ln for ln in cleaned if ln.strip()]).strip()

def _tone_open(text: str, lang: str, factoid: bool) -> str:
    t = text.strip()
    if not t:
        return {
            "ar": "مرحبًا! كيف أقدر أساعدك في رحلتك إلى مصر؟",
            "es": "¡Hola! ¿En qué puedo ayudarte con tu viaje a Egipto?",
            "en": "Hi! How can I help with your Egypt trip?",
        }.get(lang, "Hi! How can I help?")
    if factoid:
        return t
    openings = {"ar": "بكل سرور — ", "es": "¡Claro! ", "en": "Sure — "}
    if not t.lower().startswith(tuple(s.lower() for s in openings.values())):
        t = openings.get(lang, "Sure — ") + t
    return re.sub(r"\n{3,}", "\n\n", t)

def _post_fix_facts(text: str, lang: str) -> str:
    if lang == "ar":
        text = re.sub(r"\bالمملكة\s+المصرية\b", "جمهورية مصر العربية", text)
    elif lang == "es":
        text = re.sub(r"\bReino\s+de\s+Egipto\b", "República Árabe de Egipto", text, flags=re.I)
    else:
        text = re.sub(r"\bKingdom of Egypt\b", "Arab Republic of Egypt", text, flags=re.I)
    return text

print("✅ All utility functions restored successfully")


In [ ]:
# ===== Cell 2 — Pricing slots + smart follow-up =====

_MONTHS = {
    "january":"jan","february":"feb","march":"mar","april":"apr","may":"may","june":"jun",
    "july":"jul","august":"aug","september":"sep","october":"oct","november":"nov","december":"dec",
    "enero":"jan","febrero":"feb","marzo":"mar","abril":"apr","mayo":"may","junio":"jun",
    "julio":"jul","agosto":"aug","septiembre":"sep","setiembre":"sep","octubre":"oct","noviembre":"nov","diciembre":"dec",
    "يناير":"jan","فبراير":"feb","مارس":"mar","ابريل":"apr","أبريل":"apr","مايو":"may","يونيو":"jun",
    "يوليو":"jul","اغسطس":"aug","أغسطس":"aug","سبتمبر":"sep","اكتوبر":"oct","أكتوبر":"oct",
    "نوفمبر":"nov","ديسمبر":"dec"
}

def _extract_pricing_slots(q: str):
    s = q.lower()

    pax = None
    m = re.search(r"\b(\d{1,3})\s*(?:persons|people|pax|travellers|travelers|adultos|personas|أشخاص|شخص|مسافرين)\b", s)
    if m:
        pax = int(m.group(1))
    else:
        m2 = re.search(r"\bfor\s+(\d{1,3})\b", s)
        if m2:
            pax = int(m2.group(1))

    cat = None
    if "gold" in s or "غولد" in s: cat = "gold"
    if "diamond" in s or "دايموند" in s: cat = "diamond"

    month = None
    for k, v in _MONTHS.items():
        if k in s:
            month = v
            break

    duration = None
    if ("week" in s) or ("7-night" in s) or ("7 nights" in s) or ("semana" in s) or ("أسبوع" in s):
        duration = "week"

    program = None
    for name in [
        "el viaje de los faraones",
        "el rey tut",
        "misterios del nilo",
        "crónicas de el cairo",
        "encantado por el nilo",
        "maravillas de egipto",
        "la majestad del nilo",
        "esplendores de egipto",
    ]:
        if name in s:
            program = name
            break

    return {"pax": pax, "cat": cat, "month": month, "duration": duration, "program": program}

def _enough_for_direct_price(slots: dict) -> bool:
    return bool(slots.get("pax") and slots.get("cat") and slots.get("month") and slots.get("duration"))

def _pricing_query(slots: dict) -> str:
    month = slots.get("month","")
    cat = slots.get("cat","")
    dur = slots.get("duration","")
    prog = slots.get("program","")

    bits = []
    if prog: bits.append(prog)

    if dur == "week":
        bits += ["8 días", "7 noches", "8 dias", "7 noches", "7N", "8D", "week", "semana"]

    if month == "oct":
        bits += ["octubre", "october", "Oct–Apr", "Oct-Apr", "temporada", "invierno", "winter"]

    # كلمات بتظهر جنب الأسعار في الداتا غالباً
    bits += ["por persona", "per person", "USD", "Gold", "Diamond", "precio", "price", "desde", "total"]

    # boost category
    bits += [cat, cat.upper()]

    return " ".join(bits)

def _pricing_followup(lang: str, slots: dict=None) -> str:
    slots = slots or {}
    missing = []
    if not slots.get("pax"): missing.append({"ar":"عدد المسافرين","es":"nº de viajeros","en":"traveler count"}[lang])
    if not slots.get("cat"): missing.append({"ar":"الفئة (Gold/Diamond)","es":"categoría (Gold/Diamond)","en":"category (Gold/Diamond)"}[lang])
    if not slots.get("month"): missing.append({"ar":"الشهر/التواريخ","es":"mes/fechas","en":"month/dates"}[lang])
    if not slots.get("duration"): missing.append({"ar":"المدة (أسبوع/عدد الليالي)","es":"duración (semana/noches)","en":"duration (week/nights)"}[lang])

    key = missing[0] if missing else None

    if lang == "ar":
        base = "تمام. عشان أطلع لك سعر نهائي بدقة"
        return _normalize_question(f"{base}، ممكن تأكدلي {key}؟" if key else "هل تود أن أضيف تفاصيل محددة؟", "ar")
    if lang == "es":
        base = "Perfecto. Para darte un precio final con precisión"
        return _normalize_question(f"{base}, ¿me confirmas {key}?" if key else "¿Quieres que añada algún detalle en concreto?", "es")
    base = "Great. To give you an accurate final price"
    return _normalize_question(f"{base}, can you confirm {key}?" if key else "Would you like me to add any specific details?", "en")


In [ ]:
# ===== NEW Cell — Better pricing intent (Trip pricing only) =====

def _is_entry_fee_question(q: str) -> bool:
    s = q.lower()
    return any(k in s for k in [
        "entry", "ticket", "admission", "entrance", "fees", "entry price",
        "سعر الدخول", "تذاكر", "دخول", "رسوم دخول", "رسوم",
        "entrada", "boleto", "ticket", "tarifa de entrada"
    ]) and any(k in s for k in ["convert", "exchange", "rate", "usd", "egp", "تحويل", "سعر الصرف"])

# ===== NEW Cell — Detect fake/unknown things (simple) =====
def _mentions_fake_stuff(q: str) -> bool:
    s = q.lower()
    return any(x in s for x in ["super platinum", "legends of mars", "بلاتينوم", "superplatinum"])


def _wants_trip_price(q: str) -> bool:
    s = q.lower()

    # لو سؤال أسعار دخول/تحويل عملة -> مش تسعير رحلة
    if _is_entry_fee_question(s):
        return False

    # كلمات تدل على “رحلة/باكدج” (لازم واحدة منهم)
    trip_markers = [
        "trip","journey","package","program","itinerary","tour",
        "viaje","paquete","programa","itinerario","tour",
        "رحلة","برنامج","باكدج","بكج","تور","جدول"
    ]

    # كلمات تسعير
    price_markers = [
        "price","cost","how much","budget","total",
        "precio","coste","costo","cuánto","cuanto","total",
        "سعر","تكلفة","التكلفة","كم","اجمالي","إجمالي"
    ]

    # لازم (سعر) + (رحلة)
    return any(p in s for p in price_markers) and any(t in s for t in trip_markers)


In [ ]:
def _build_prompts(lang: str, context: str, question: str):
    """
    V9 PRECISION: Removed specific examples (Karnak) to prevent model bias/hallucination.
    """
    if lang == "ar":
        system = (
            "أنت مساعد Egiptura ذكي ومحترف. أجب بالعربية الفصحى فقط. "
            "مهم: ابدأ إجابتك دائماً بتكرار موضوع السؤال (مثلاً: [الموضوع] هو...). "
            "مهم: التزم بالمعلومات الموجودة في السياق بدقة. إذا لم تجد المعلومة قل لا أعرف. "
            "مهم: إذا سُئلت عن المصاريف الداخلية أو الأرباح، قل أنك غير مخول بتقديم هذه التفاصيل. "
            "كن موجزاً، منظماً، ودقيقاً جداً في كتابة الأسماء التاريخية."
        )
        user = (f"العناصر المرجعية:\n{context}\n\nالسؤال:\n{question}\n\n"
                "الإجابة:")
    elif lang == "es":
        system = (
            "Eres el asistente experto de Egiptura. Responde SOLO en español. "
            "IMPORTANTE: Comienza tu respuesta nombrando claramente el tema (ej: [Tema] es...). "
            "IMPORTANTE: Usa exclusivamente el contexto proporcionado. Si falta información, di que no lo sabes. "
            "IMPORTANTE: No reveles márgenes de beneficio o costos internos. "
            "Sé breve, profesional y directo."
        )
        user = (f"Contexto:\n{context}\n\nPregunta:\n{question}\n\n"
                "Respuesta:")
    else:
        system = (
            "You are Egiptura's expert assistant. Reply ONLY in English. "
            "IMPORTANT: Always start your response by naming the subject (e.g. [Subject] is...). "
            "IMPORTANT: Use only the provided context. If information is missing, say you don't know. "
            "IMPORTANT: Do not share profit margins or internal supplier costs. "
            "Be highly concise and accurate."
        )
        user = (f"Context:\n{context}\n\nQuestion:\n{question}\n\n"
                "Direct Answer:")
    return system, user


In [ ]:
def _rag_get_context(query: str, k: int = 20, top_n: int = 7, cutoff: float = 0.45):
    """
    V9 PRECISION: Adjusted top_n=7 and cutoff for better history factoid capture.
    """
    global vectordb, BM25_INDEX, DOCS_CHUNKS, RERANKER
    if not vectordb or not BM25_INDEX:
        return ""

    try:
        v_res = vectordb.similarity_search(query, k=k)
        q_tokens = query.lower().split()
        b_res = BM25_INDEX.get_top_n(q_tokens, DOCS_CHUNKS, n=k)

        seen = set()
        candidates = []
        for d in v_res + b_res:
            txt = d.page_content.strip()
            if txt not in seen:
                candidates.append(d)
                seen.add(txt)

        if RERANKER and candidates:
            pairs = [(query, d.page_content) for d in candidates]
            scores = RERANKER.predict(pairs)
            scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
            # V9: Slightly more permissive cutoff (-7.0 instead of -6.0) for breadth
            topdocs = [d for d, s in scored[:top_n] if s > -7.0]
        else:
            topdocs = candidates[:top_n]

        ctx_lines = []
        for d in topdocs:
            meta = d.metadata
            src_name = f"{meta.get('source','')}:p{meta.get('page','')}"
            txt = re.sub(r"\s{2,}", " ", d.page_content.strip())
            ctx_lines.append(f"[{src_name}] {txt}")

        return "\n\n".join(ctx_lines)

    except Exception as e:
        print(f"[RAG error] {e}")
        return ""


In [ ]:
# ===== PRICE_TABLE + Resolvers (RESTORED) =====
import re

PRICE_TABLE = {
    ("rey tut", "gold", "summer"):    {"single": 1365, "double": 1145, "triple": 1100},
    ("rey tut", "gold", "winter"):    {"single": 1465, "double": 1225, "triple": 1185},
    ("rey tut", "diamond", "summer"): {"single": None, "double": 1585, "triple": None},
    ("rey tut", "diamond", "winter"): {"single": None, "double": 1690, "triple": None},

    ("cronicas", "gold", "summer"):    {"single": 730,  "double": 580, "triple": 540},
    ("cronicas", "gold", "winter"):    {"single": 830,  "double": 640, "triple": 600},
    ("cronicas", "diamond", "summer"): {"single": 1140, "double": 820, "triple": 780},
    ("cronicas", "diamond", "winter"): {"single": 1240, "double": 880, "triple": 840},

    ("faraones", "gold", "summer"):    {"single": 1210, "double": 965,  "triple": 915},
    ("faraones", "gold", "winter"):    {"single": 1310, "double": 1025, "triple": 975},
    ("faraones", "diamond", "summer"): {"single": 1580, "double": 1125, "triple": 1075},
    ("faraones", "diamond", "winter"): {"single": 1700, "double": 1200, "triple": 1150},

    ("encantado", "gold", "summer"):    {"single": 1440, "double": 985,  "triple": 935},
    ("encantado", "gold", "winter"):    {"single": 1540, "double": 1040, "triple": 990},
    ("encantado", "diamond", "summer"): {"single": 1800, "double": 1195, "triple": 1145},
    ("encantado", "diamond", "winter"): {"single": 1900, "double": 1255, "triple": 1205},

    ("misterios", "gold", "summer"):    {"single": 1440, "double": 985,  "triple": 935},
    ("misterios", "gold", "winter"):    {"single": 1540, "double": 1040, "triple": 990},
    ("misterios", "diamond", "summer"): {"single": 1800, "double": 1195, "triple": 1145},
    ("misterios", "diamond", "winter"): {"single": 1900, "double": 1255, "triple": 1205},

    ("esplendores", "gold", "summer"):    {"single": 1440, "double": 985,  "triple": 935},
    ("esplendores", "gold", "winter"):    {"single": 1540, "double": 1040, "triple": 990},
    ("esplendores", "diamond", "summer"): {"single": 1800, "double": 1195, "triple": 1145},
    ("esplendores", "diamond", "winter"): {"single": 1900, "double": 1255, "triple": 1205},

    ("majestad", "gold", "summer"):    {"single": 1745, "double": 1155, "triple": 1105},
    ("majestad", "gold", "winter"):    {"single": 1865, "double": 1220, "triple": 1165},
    ("majestad", "diamond", "summer"): {"single": 1980, "double": 1345, "triple": 1295},
    ("majestad", "diamond", "winter"): {"single": 2100, "double": 1405, "triple": 1345},

    ("maravillas", "gold", "summer"):    {"single": 1825, "double": 1225, "triple": 1175},
    ("maravillas", "gold", "winter"):    {"single": 1950, "double": 1295, "triple": 1240},
    ("maravillas", "diamond", "summer"): {"single": 2100, "double": 1375, "triple": 1325},
    ("maravillas", "diamond", "winter"): {"single": 2250, "double": 1445, "triple": 1395},

    ("ecos", "gold", "summer"):    {"single": 1675, "double": 1145, "triple": 1090},
    ("ecos", "gold", "winter"):    {"single": 1800, "double": 1195, "triple": 1140},
    ("ecos", "diamond", "summer"): {"single": 1925, "double": 1275, "triple": 1225},
    ("ecos", "diamond", "winter"): {"single": 2050, "double": 1345, "triple": 1295},

    ("leyendas", "gold", "summer"):    {"single": 1475, "double": 975,  "triple": 920},
    ("leyendas", "gold", "winter"):    {"single": 1575, "double": 1025, "triple": 970},
    ("leyendas", "diamond", "summer"): {"single": 1675, "double": 1075, "triple": 1025},
    ("leyendas", "diamond", "winter"): {"single": 1775, "double": 1145, "triple": 1090},
}

PROGRAM_ALIASES = {
    "rey tut":    ["el rey tut", "rey tut", "king tut", "tut", "gem del rey"],
    "cronicas":   ["crónicas", "cronicas", "el cairo", "cairo chronicles"],
    "faraones":   ["faraones", "pharaohs", "viaje de los faraones", "el viaje"],
    "encantado":  ["encantado", "enchanted", "encantando", "encantado por el nilo"],
    "misterios":  ["misterios", "mysteries", "misterios del nilo"],
    "esplendores":["esplendores", "splendors", "esplendores de egipto"],
    "majestad":   ["majestad", "majesty", "la majestad", "majestad del nilo"],
    "maravillas": ["maravillas", "wonders", "maravillas de egipto"],
    "ecos":       ["ecos", "echoes", "ecos de la eternidad", "eternidad"],
    "leyendas":   ["leyendas", "legends", "leyendas del nilo"],
}

PROGRAM_FULL_NAMES = {
    "rey tut":    "El Rey Tut te invita",
    "cronicas":   "Crónicas de El Cairo",
    "faraones":   "El Viaje de los Faraones",
    "encantado":  "Encantado por el Nilo",
    "misterios":  "Misterios del Nilo",
    "esplendores":"Esplendores de Egipto",
    "majestad":   "La Majestad del Nilo",
    "maravillas": "Maravillas de Egipto",
    "ecos":       "Ecos de la Eternidad",
    "leyendas":   "Leyendas del Nilo",
}

SUMMER_MONTHS = {"may", "mayo", "jun", "june", "junio", "jul", "july", "julio",
                 "aug", "august", "agosto", "sep", "sept", "september", "septiembre"}
WINTER_MONTHS = {"oct", "october", "octubre", "nov", "november", "noviembre",
                 "dec", "december", "diciembre", "jan", "january", "enero",
                 "feb", "february", "febrero", "mar", "march", "marzo",
                 "apr", "april", "abril"}

def _resolve_program(q: str):
    s = q.lower()
    for key, aliases in PROGRAM_ALIASES.items():
        if any(a in s for a in aliases):
            return key
    return None

def _resolve_season(q: str):
    s = q.lower()
    if any(m in s for m in WINTER_MONTHS): return "winter"
    if any(m in s for m in SUMMER_MONTHS): return "summer"
    return None

def _resolve_category(q: str):
    s = q.lower()
    if "diamond" in s: return "diamond"
    if "gold" in s: return "gold"
    return None

def _resolve_room(q: str):
    s = q.lower()
    if any(x in s for x in ["single", "individual", "sencilla", "solo", "1 person"]): return "single"
    if any(x in s for x in ["triple", "3 person"]): return "triple"
    return "double"

def _resolve_pax(q: str):
    m = re.search(r'(\d+)\s*(?:persons?|people|pax|viajeros?|personas?|travelers?)', q.lower())
    if m: return int(m.group(1))
    m2 = re.search(r'\bfor\s+(\d{1,3})\b', q.lower())
    if m2: return int(m2.group(1))
    return None

print("PRICE_TABLE loaded:", len(PRICE_TABLE), "entries")
print("Sample: Faraones Gold Winter Double =", PRICE_TABLE.get(("faraones", "gold", "winter"), {}).get("double"))


In [ ]:
def _build_price_context(q: str) -> str:
    """GROUND TRUTH: Python-calculated prices to prevent LLM math errors."""
    s = q.lower()
    season = _resolve_season(q)
    cat = _resolve_category(q)
    room = _resolve_room(q)
    pax = _resolve_pax(q) or 1
    if not (season and cat): return ''
    prog_key = _resolve_program(q)
    lines = []
    sl = 'High Season (Oct-Apr)' if season == 'winter' else 'Low Season (May-Sep)'
    if prog_key:
        key = (prog_key, cat, season)
        prices = PRICE_TABLE.get(key)
        if prices:
            name = PROGRAM_FULL_NAMES[prog_key]
            pp = prices.get(room, prices['double'])
            total = pp * pax
            lines.append(f'- {name}: USD {pp}/person ({room}) | TOTAL for {pax} pax: USD {total:,}')
    else:
        for p in ['faraones', 'encantado', 'rey tut']:
            key = (p, cat, season)
            prices = PRICE_TABLE.get(key)
            if prices:
                name = PROGRAM_FULL_NAMES[p]
                pp = prices.get(room, prices['double'])
                lines.append(f'- {name}: USD {pp}/person | TOTAL for {pax} pax: USD {pp*pax:,}')
    if not lines: return ''
    header = f'[GROUND TRUTH PRICES - {cat.upper()} - {sl}]'
    return header + chr(10) + chr(10).join(lines)


In [ ]:
def answer(question: str) -> str:
    """
    V9 PRECISION: Enhanced Pricing List support + Internal Cost Shielding.
    """
    global tok, tokenizer, model
    tok = tok if 'tok' in globals() else tokenizer
    user_q = question.strip()
    ql = user_q.lower()
    lang = _detect_lang(user_q)

    if _is_greeting(user_q):
        return {'ar': 'مرحبا! كيف أقدر أساعدك في رحلتك لمصر؟',
                'es': '¡Hola! ¿Cómo puedo ayudarte con tu viaje a Egipto?',
                'en': 'Hi!'}.get(lang, 'Hi!')

    # 1. INTERNAL COST SHIELD (Stricter Detect)
    if any(x in ql for x in ["profit", "margin", "internal cost", "supplier", "markup", "ربح", "هامش الرقح", "margen", "beneficio"]):
        return "I am not authorized to disclose internal costs or profit margins. I can only provide the final package price to the client."

    # 2. RESOLVE ENTITIES
    season = _resolve_season(user_q)
    cat = _resolve_category(user_q)
    prog_key = _resolve_program(user_q)
    room = _resolve_room(user_q)
    pax = _resolve_pax(user_q) or 1

    # 3. DUAL-PATH PRICING
    has_price_kw = any(k in ql for k in ['price','cost','how much','total','quote','cuanto','precio','coste', 'سعر', 'كم'])
    wants_all = any(x in ql for x in ['3 programs', 'all options', 'compare', 'خيارات', 'opciones'])
    is_pricing = _wants_trip_price(user_q) or (has_price_kw and (cat or prog_key))

    if is_pricing and cat and season:
        # PATH A: Single specific program
        if prog_key and not wants_all:
            key = (prog_key, cat, season)
            prices = PRICE_TABLE.get(key)
            if prices:
                name = PROGRAM_FULL_NAMES[prog_key]
                pp = prices.get(room, prices['double'])
                sl = 'October-April' if season == 'winter' else 'May-September'
                return (f"{name} ({cat.capitalize()}, {sl}):\n"
                        f"- Price per person: USD {pp:,} ({room} occupancy)\n"
                        f"- Total for {pax} travelers: USD {pp*pax:,}\n\n"
                        f"Let me know if you would like the full itinerary details.")

        # PATH B: List all 3 main programs (Common request)
        options = []
        target_progs = ['faraones', 'encantado', 'rey tut']
        for p in target_progs:
            key = (p, cat, season)
            prices = PRICE_TABLE.get(key)
            if prices:
                name = PROGRAM_FULL_NAMES[p]
                pp = prices.get(room, prices['double'])
                options.append(f"- {name}: USD {pp:,}/person | Total: USD {pp*pax:,}")

        if options:
            sl = 'October-April' if season == 'winter' else 'May-September'
            return f"Here are the prices for our 3 main programs ({cat.capitalize()}, {sl}, {room} occupancy):\n\n" + "\n".join(options) + "\n\nWhich one would you like to explore?"

    # 4. RAG PATH
    fact = _is_factoid(user_q)
    context = _rag_get_context(user_q)

    # EXCHANGE RATE INJECTION
    if any(x in ql for x in ["entry", "fee", "ticket", "entrance", "رسم", "تذكرة", "entrada"]):
        context = "[POLICY] Fixed Exchange Rate: 1 USD = 45 EGP for all entry fees mentioned in documents.\n\n" + context

    if not context.strip() and fact:
        return "I'm sorry, I don't have information about that in my records."

    system, user_body = _build_prompts(lang, context, user_q)

    # MODEL CALL
    use_template = hasattr(tok, 'apply_chat_template') and getattr(tok, 'chat_template', None)
    if use_template:
        messages = [{'role':'system','content':system},{'role':'user','content':user_body}]
        enc = tok.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
        ins = {'input_ids': enc, 'attention_mask': torch.ones_like(enc)}
    else:
        prompt = f'{system}\n\nUSER:\n{user_body}\n\nASSISTANT:\n'
        enc = tok(prompt, return_tensors='pt').to(model.device)
        ins = {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']}

    with torch.inference_mode():
        out = model.generate(**ins, max_new_tokens=128, do_sample=False, repetition_penalty=1.1, eos_token_id=tok.eos_token_id)

    text = tok.decode(out[0, ins['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

    # POST-PROCESS
    text = _clean(text)
    text = _enforce_lang(text, lang)
    text = _post_fix_facts(text, lang)
    text = _tone_open(text, lang, factoid=fact)

    return text.strip()


In [ ]:
# ===== Cell 6 — Quick test =====
q = "how much the journey in egypt for week cost in october with gold category for 10 persons?"
print(answer(q))


In [ ]:
def normalize_text(text: str) -> str:
    if text is None: return ""
    s = str(text).lower().strip()
    s = strip_accents(s)
    s = s.translate(SPANISH_CHAR_MAP)
    s = s.translate(ARABIC_CHAR_MAP)

    # Normalization for prices
    s = s.replace('usd', ' ').replace('$', ' ').replace(',', '')
    s = re.sub(r'\s+', ' ', s)

    # V9 HISTORY SYNONYMS (Prevents evaluation misses on typos)
    synonyms = {
        "الكرنيوك": "الكرنك",
        "الكركن": "الكرنك",
        "حتشپسوت": "حتشبسوت",
        "حتشبسوط": "حتشبسوت",
        "الملوكو": "الملوك",
        "الملوق": "الملوك",
        "بو الهول": "أبو الهول",
        "أبو هول": "أبو الهول",
        "هيروغرافية": "هيروغليفية",
        "هيروجليفية": "هيروغليفية",
    }
    for k, v in synonyms.items():
        s = s.replace(k, v)

    for k, v in GENERIC_REPLACEMENTS.items():
        s = s.replace(k, v)

    return s.strip()


In [ ]:
# ===== FULL EVALUATION (with EVAL dataset) =====
import re, time
import pandas as pd

EVAL = [
{"category":"pricing","q":"how much the journey in egypt for week cost in october with gold category for 10 persons?","expected_numbers":["1025","1040","1225","10250","10400","12250"],"expected_keywords_any":["Gold","October","USD","El Rey Tut","Encantado por el Nilo","El Viaje de los Faraones"]},
{"category":"pricing","q":"Give me 3 options for a 7-night Egypt trip in October (Gold). Show per-person price in USD.","expected_numbers":["1025","1040","1225"],"expected_keywords_any":["Gold","October","USD"]},
{"category":"pricing","q":"For 10 travelers in October (Gold), what is the total cost for El Viaje de los Faraones?","expected_numbers":["1025","10250"],"expected_keywords_any":["El Viaje de los Faraones","Gold","October","USD"]},
{"category":"pricing","q":"For 10 travelers in October (Gold), what is the total cost for El Rey Tut?","expected_numbers":["1225","12250"],"expected_keywords_any":["El Rey Tut","Gold","October","USD"]},
{"category":"pricing","q":"For 10 travelers in October (Gold), what is the total cost for Encantado por el Nilo?","expected_numbers":["1040","10400"],"expected_keywords_any":["Encantado por el Nilo","Gold","October","USD"]},
{"category":"pricing","q":"If we are 2 travelers in October (Gold), what is the per-person price for El Rey Tut?","expected_numbers":["1225"],"expected_keywords_any":["El Rey Tut","Gold","October"]},
{"category":"pricing","q":"If we are 4 travelers in October (Gold), what is the per-person price for Encantado por el Nilo?","expected_numbers":["1040"],"expected_keywords_any":["Encantado por el Nilo","Gold","October"]},
{"category":"pricing","q":"List the per-person prices for: El Viaje de los Faraones, Encantado por el Nilo, El Rey Tut (October, Gold).","expected_numbers":["1025","1040","1225"],"expected_keywords_any":["El Viaje de los Faraones","Encantado por el Nilo","El Rey Tut","October","Gold"]},
{"category":"pricing","q":"What is the group total for 10 persons in October (Gold) for the 3 week programs?","expected_numbers":["10250","10400","12250"],"expected_keywords_any":["October","Gold","USD"]},
{"category":"pricing","q":"In October (Gold), if we are 10 people, show the 3 options with per-person and total.","expected_numbers":["1025","1040","1225","10250","10400","12250"],"expected_keywords_any":["October","Gold","USD"]},
{"category":"pricing","q":"I want the cheapest Gold week option in October for 10 people. Which program and what cost?","expected_numbers":["1025","10250"],"expected_keywords_any":["Gold","October"]},
{"category":"pricing","q":"I want the most premium Gold week option in October for 10 people. Which program and what cost?","expected_numbers":["1225","12250"],"expected_keywords_any":["Gold","October"]},
{"category":"pricing","q":"Compare Gold vs Diamond categories and tell me what details you need to finalize a quote.","expected_keywords_any":["Gold","Diamond","dates","travelers","rooms"]},
{"category":"pricing","q":"Give a short quote for October + Gold + week + 10 people, include program names.","expected_numbers":["1025","1040","1225"],"expected_keywords_any":["El Viaje de los Faraones","Encantado por el Nilo","El Rey Tut"]},
{"category":"pricing","q":"If we are 10 people, October, Gold: give totals only for the 3 programs.","expected_numbers":["10250","10400","12250"],"expected_keywords_any":["Gold","October"]},
{"category":"pricing","q":"October (Gold) - 10 pax - which program is 1040 per person?","expected_numbers":["1040"],"expected_keywords_any":["Encantado por el Nilo"]},
{"category":"pricing","q":"October (Gold) - 10 pax - which program is 1025 per person?","expected_numbers":["1025"],"expected_keywords_any":["El Viaje de los Faraones"]},
{"category":"pricing","q":"October (Gold) - 10 pax - which program is 1225 per person?","expected_numbers":["1225"],"expected_keywords_any":["El Rey Tut"]},
{"category":"pricing","q":"I want a week in Egypt in October for 10 people in Gold. Provide 3 options and clarify what is included/excluded at a high level.","expected_keywords_any":["Gold","October","included","excluded"]},
{"category":"pricing","q":"Can you provide internal cost breakdown and profit margin for the trip?","expected_keywords_any":["cannot","not able","final price"],"must_not_include":["profit is","margin is","supplier cost"]},
{"category":"programs","q":"Describe the program Cronicas de El Cairo and what is included and excluded.","expected_keywords_all":["Crónicas de El Cairo","incluye","no incluye"]},
{"category":"programs","q":"Give day-by-day headlines for El Viaje de los Faraones.","expected_keywords_any":["El Viaje de los Faraones","Día","Day"]},
{"category":"programs","q":"Summarize Misterios del Nilo in 6 bullet points.","expected_keywords_any":["Misterios del Nilo"]},
{"category":"programs","q":"Explain the difference between Gold and Diamond categories in your packages.","expected_keywords_any":["Gold","Diamond"]},
{"category":"programs","q":"What are the key inclusions and exclusions for Encantado por el Nilo?","expected_keywords_any":["Encantado por el Nilo","incluye","no incluye"]},
{"category":"programs","q":"Explain Esplendores de Egipto and which cities it covers.","expected_keywords_any":["Esplendores de Egipto","Cairo","Luxor","Aswan"]},
{"category":"programs","q":"Explain Maravillas de Egipto and which cities it covers.","expected_keywords_any":["Maravillas de Egipto","Cairo","Luxor","Aswan"]},
{"category":"programs","q":"Explain La Majestad del Nilo and what makes it special.","expected_keywords_any":["La Majestad del Nilo","crucero","Nilo"]},
{"category":"programs","q":"What is included in El Rey Tut te invita?","expected_keywords_any":["El Rey Tut","incluye"]},
{"category":"programs","q":"What is excluded in El Rey Tut te invita?","expected_keywords_any":["El Rey Tut","no incluye"]},
{"category":"programs","q":"What is the program Misterios del Nilo focus: cruise, cities, or desert?","expected_keywords_any":["Nilo","crucero"]},
{"category":"programs","q":"List the main highlights/attractions usually covered in your week programs.","expected_keywords_any":["Giza","Pyramids","Luxor","Aswan"]},
{"category":"programs","q":"If I want a custom trip, what info do you need from me?","expected_keywords_any":["travelers","dates","category","interests","budget"]},
{"category":"programs","q":"Do your programs include private transfers? Answer based on your service style.","expected_keywords_any":["private","transfer"]},
{"category":"programs","q":"What kind of travelers are your Gold packages best for?","expected_keywords_any":["Gold"]},
{"category":"policies","q":"What is the cancellation and refund policy? Summarize clearly.","expected_keywords_any":["cancel","refund","policy"]},
{"category":"policies","q":"How do deposits work and when is the final payment due?","expected_keywords_any":["deposit","final payment","due"]},
{"category":"policies","q":"What payment methods do you accept?","expected_keywords_any":["payment","card","transfer","PayPal","Stripe"]},
{"category":"policies","q":"Do you include international flights in your packages?","expected_keywords_any":["international flights","not included","excluded"]},
{"category":"policies","q":"What documents do travelers need for Egypt (passport/visa) according to your FAQ?","expected_keywords_any":["passport","visa"]},
{"category":"policies","q":"Do you provide travel insurance? What do you recommend?","expected_keywords_any":["insurance","recommend"]},
{"category":"policies","q":"Can I change the itinerary after booking? What is the rule?","expected_keywords_any":["change","subject","availability"]},
{"category":"policies","q":"Are tips included? Explain your policy.","expected_keywords_any":["tips","not included","optional"]},
{"category":"policies","q":"What is your policy on missed services due to late arrival?","expected_keywords_any":["late","arrival"]},
{"category":"policies","q":"What are the main terms and conditions before booking?","expected_keywords_any":["terms","conditions","booking"]},
{"category":"policies","q":"Do you offer guided tours in Spanish?","expected_keywords_any":["Spanish","guía"]},
{"category":"policies","q":"How do you handle special requests (dietary, mobility, kids)?","expected_keywords_any":["special","request"]},
{"category":"rules","q":"I want a Nile cruise but I dont know which visits are included. Suggest the default visits.","expected_keywords_all":["Karnak","Luxor Temple","Valley of the Kings","Colossi of Memnon","Kom Ombo","Edfu","High Dam","Philae"]},
{"category":"rules","q":"We will be in Cairo only. What visits will you include by default if I dont specify?","expected_keywords_any":["Pyramids","Giza"]},
{"category":"rules","q":"We want Cairo and we already included the Pyramids. Please do NOT add it again. Confirm no duplication.","expected_keywords_any":["no duplication","already included","not add again"]},
{"category":"rules","q":"Show the default Nile cruise visits clearly as a list (Luxor, Kom Ombo, Edfu, Aswan).","expected_keywords_any":["Luxor","Kom Ombo","Edfu","Aswan"]},
{"category":"rules","q":"Can you add an extra visit to Dendera during the cruise route? How would you handle it?","expected_keywords_any":["add","possible","extra","arrange"]},
{"category":"rules","q":"Remove Edfu temple from the default cruise visits. How would you handle it?","expected_keywords_any":["remove","adjust","update"]},
{"category":"rules","q":"If the client requests Cairo without details, what key visit must be included?","expected_keywords_any":["Pyramids"]},
{"category":"rules","q":"If the client specifies Pyramids already, what should you avoid?","expected_keywords_any":["avoid","duplication"]},
{"category":"rules","q":"Client wants only cruise + no temples. How do you respond politely?","expected_keywords_any":["possible","adjust","customize"]},
{"category":"rules","q":"Explain how you confirm included visits to avoid misunderstandings.","expected_keywords_any":["confirm","included","list"]},
{"category":"transport","q":"Show the transportation pricing table for Cairo and explain that the cost is split among travelers.","expected_keywords_any":["Cairo","split","travelers","divided"]},
{"category":"transport","q":"Show transportation pricing for Luxor (vehicle types + passenger counts).","expected_keywords_any":["Luxor"]},
{"category":"transport","q":"Show transportation pricing for Aswan (vehicle types + passenger counts).","expected_keywords_any":["Aswan"]},
{"category":"transport","q":"Which vehicle should we use for 6 travelers in Cairo, and how is cost shared?","expected_keywords_any":["6","vehicle","split"]},
{"category":"transport","q":"Explain private transfer vs shared transportation in your service style.","expected_keywords_any":["private","transfer"]},
{"category":"transport","q":"If we are 2 travelers, what vehicle type is typical for Cairo transfers?","expected_keywords_any":["car","sedan","private"]},
{"category":"transport","q":"If we are 10 travelers, what vehicle type is typical for transfers?","expected_keywords_any":["van","minibus","bus"]},
{"category":"transport","q":"Does transport price change by passenger count? Explain.","expected_keywords_any":["passenger","count"]},
{"category":"transport","q":"Do you include airport transfers?","expected_keywords_any":["airport","transfer"]},
{"category":"transport","q":"Explain how transport is divided among travelers in simple words.","expected_keywords_any":["divided","travelers","share"]},
{"category":"traps","q":"Do you offer a Super Platinum category and how much does it cost?","expected_keywords_any":["Gold","Diamond","not available","do not"],"must_not_include":["Super Platinum costs","price is"]},
{"category":"traps","q":"What is the exact price of an imaginary program called Legends of Mars?","expected_keywords_any":["not found","not available","cannot"],"must_not_include":["Legends of Mars costs","price is"]},
{"category":"traps","q":"Do you include free business-class international flights in every package?","expected_keywords_any":["not included","international flights"]},
{"category":"traps","q":"Is the Pyramids visit optional in Cairo even if the client doesnt specify?","expected_keywords_any":["default","include","Pyramids"]},
{"category":"traps","q":"Can you guarantee 100% refund anytime?","expected_keywords_any":["policy","refund"],"must_not_include":["100% refund anytime","guaranteed"]},
{"category":"traps","q":"Tell me the price of a 30-day Mars tour in Egypt.","expected_keywords_any":["not available","cannot"]},
{"category":"traps","q":"Do you have a program called Faraones 2.0 Ultra?","expected_keywords_any":["not available","not found","cannot"]},
{"category":"style","q":"Answer in Spanish: What is Egiptura and what makes it luxury?","expected_keywords_any":["Egiptura"]},
{"category":"style","q":"Answer in Arabic: هل يمكنني تصميم برنامج خاص؟ وكيف تبدأون التخطيط؟","expected_keywords_any":["برنامج","التخطيط"]},
{"category":"style","q":"Answer in English: Give a 3-line intro about Egiptura services.","expected_keywords_any":["Egiptura","Egypt"]},
{"category":"history","q":"من هو الملك توت عنخ آمون ولماذا اشتهر؟","expected_keywords_any":["توت","مقبرة","وادي الملوك"]},
{"category":"history","q":"إيه الفرق بين الأهرامات في الجيزة وأهرامات سقارة؟","expected_keywords_any":["الجيزة","سقارة"]},
{"category":"history","q":"اشرح بإيجاز أهمية معبد الكرنك تاريخيا.","expected_keywords_any":["الكرنك","الأقصر"]},
{"category":"history","q":"ما هو وادي الملوك ولماذا بني؟","expected_keywords_any":["وادي الملوك"]},
{"category":"history","q":"من هي الملكة حتشبسوت وما أشهر إنجازاتها؟","expected_keywords_any":["حتشبسوت"]},
{"category":"history","q":"ما قصة معبد أبو سمبل ولماذا تم نقله؟","expected_keywords_any":["أبو سمبل","النقل"]},
{"category":"history","q":"اشرح بإيجاز معنى التحنيط عند المصريين القدماء.","expected_keywords_any":["تحنيط"]},
{"category":"history","q":"ما هو كتاب الموتى عند المصريين القدماء؟","expected_keywords_any":["كتاب الموتى"]},
{"category":"history","q":"من هو رمسيس الثاني ولماذا يعد من أشهر الفراعنة؟","expected_keywords_any":["رمسيس"]},
{"category":"history","q":"ما أهمية معبد فيلة (Philae)؟","expected_keywords_any":["فيلة","إيزيس"]},
{"category":"history","q":"ما هو معبد كوم أمبو ولماذا له تصميم مزدوج؟","expected_keywords_any":["كوم أمبو"]},
{"category":"history","q":"ما الذي يميز معبد إدفو؟","expected_keywords_any":["إدفو","حورس"]},
{"category":"history","q":"اشرح باختصار تاريخ مدينة الأقصر القديمة (طيبة).","expected_keywords_any":["طيبة","الأقصر"]},
{"category":"history","q":"ما الفرق بين معبد الأقصر ومعبد الكرنك؟","expected_keywords_any":["الأقصر","الكرنك"]},
{"category":"history","q":"من هم الهكسوس وما علاقتهم بتاريخ مصر؟","expected_keywords_any":["الهكسوس"]},
{"category":"history","q":"اشرح بإيجاز فترات الدولة القديمة والوسطى والحديثة.","expected_keywords_any":["الدولة القديمة","الوسطى","الحديثة"]},
{"category":"history","q":"ما هي حجر رشيد ولماذا هو مهم؟","expected_keywords_any":["حجر رشيد"]},
{"category":"history","q":"من هو شامبليون وما علاقته بالهيروغليفية؟","expected_keywords_any":["شامبليون","الهيروغليفية"]},
{"category":"history","q":"اشرح بإيجاز معنى الهيروغليفية وكيف كانت تستخدم.","expected_keywords_any":["هيروغليفية"]},
{"category":"history","q":"ما هي الأهرامات ولماذا بنيت؟","expected_keywords_any":["الأهرامات"]},
{"category":"history","q":"من هو الملك خوفو وما علاقته بالهرم الأكبر؟","expected_keywords_any":["خوفو","الهرم الأكبر"]},
{"category":"history","q":"من هو خفرع ومنكاورع؟","expected_keywords_any":["خفرع","منكاورع"]},
{"category":"history","q":"ما هو تمثال أبو الهول وما أشهر نظريات بنائه؟","expected_keywords_any":["أبو الهول"]},
{"category":"history","q":"اشرح بإيجاز تاريخ ممفيس (منف).","expected_keywords_any":["منف","ممفيس"]},
{"category":"history","q":"ما هي سقارة وما أشهر معالمها؟","expected_keywords_any":["سقارة","زوسر"]},
{"category":"history","q":"من هو زوسر وما أهمية الهرم المدرج؟","expected_keywords_any":["زوسر","الهرم المدرج"]},
{"category":"history","q":"ما هي دهشور وما الفرق بين الهرم المنحني والأحمر؟","expected_keywords_any":["دهشور","المنحني","الأحمر"]},
{"category":"history","q":"اشرح بإيجاز تاريخ الإسكندرية في العصر البطلمي.","expected_keywords_any":["الإسكندرية","بطلمي"]},
{"category":"history","q":"من هي كليوباترا ولماذا تعد شخصية محورية؟","expected_keywords_any":["كليوباترا"]},
{"category":"history","q":"ما هو الفن المصري القديم وما سماته الأساسية؟","expected_keywords_any":["الفن","المصري"]},
{"category":"history","q":"اشرح بإيجاز الديانة المصرية القديمة وأهم الآلهة.","expected_keywords_any":["آلهة","إيزيس","أوزيريس"]},
{"category":"history","q":"ما هو دور النيل في قيام الحضارة المصرية؟","expected_keywords_any":["النيل"]},
{"category":"history","q":"كيف كان نظام الحكم في مصر القديمة؟","expected_keywords_any":["فرعون","حكم"]},
{"category":"history","q":"اشرح بإيجاز معنى الماعت (Maat) في الفكر المصري القديم.","expected_keywords_any":["ماعت"]},
{"category":"history","q":"من هو أخناتون وما قصة عبادة آتون؟","expected_keywords_any":["أخناتون","آتون"]},
{"category":"history","q":"ما هي تل العمارنة ولماذا هي مهمة؟","expected_keywords_any":["العمارنة"]},
{"category":"history","q":"اشرح بإيجاز دور الكهنة في مصر القديمة.","expected_keywords_any":["الكهنة"]},
{"category":"history","q":"ما هو المتحف المصري الكبير (GEM) ولماذا مهم ثقافيا؟","expected_keywords_any":["GEM","المتحف المصري الكبير"]},
{"category":"history","q":"ما أشهر القطع المرتبطة بتوت عنخ آمون؟","expected_keywords_any":["قناع","توت"]},
{"category":"history","q":"اشرح بإيجاز الفرق بين المقبرة والمعبد الجنائزي.","expected_keywords_any":["مقبرة","معبد جنائزي"]},
{"category":"history","q":"ما قصة معبد حتشبسوت (الدير البحري)؟","expected_keywords_any":["الدير البحري","حتشبسوت"]},
{"category":"history","q":"ما هي مقابر النبلاء في الأقصر وما الذي يميزها؟","expected_keywords_any":["مقابر","النبلاء"]},
{"category":"history","q":"اشرح بإيجاز أهمية مدينة أسوان تاريخيا.","expected_keywords_any":["أسوان"]},
{"category":"history","q":"ما هو السد العالي وما تأثيره على مصر الحديثة؟","expected_keywords_any":["السد العالي"]},
{"category":"history","q":"ما هي النوبة ولماذا لها ثقافة مميزة؟","expected_keywords_any":["النوبة"]},
{"category":"history","q":"اشرح بإيجاز أسطورة إيزيس وأوزيريس.","expected_keywords_any":["إيزيس","أوزيريس"]},
]

# ===== IMPROVED FULL EVALUATION (fairer + stricter scoring) =====
import re
import time
import unicodedata
from typing import List, Dict, Any
import pandas as pd

# Keep your EVAL exactly as it is above this block.
# This script only replaces the scoring logic.

# ----------------------------
# Text normalization utilities
# ----------------------------
ARABIC_CHAR_MAP = str.maketrans({
    'أ': 'ا', 'إ': 'ا', 'آ': 'ا',
    'ى': 'ي', 'ئ': 'ي', 'ؤ': 'و',
    'ة': 'ه',
})

SPANISH_CHAR_MAP = str.maketrans({
    'á': 'a', 'à': 'a', 'ä': 'a', 'â': 'a',
    'é': 'e', 'è': 'e', 'ë': 'e', 'ê': 'e',
    'í': 'i', 'ì': 'i', 'ï': 'i', 'î': 'i',
    'ó': 'o', 'ò': 'o', 'ö': 'o', 'ô': 'o',
    'ú': 'u', 'ù': 'u', 'ü': 'u', 'û': 'u',
    'ñ': 'n',
})

GENERIC_REPLACEMENTS = {
    "egipturra": "egiptura",
    "egitpura": "egiptura",
    "egiptune": "egiptura",
    "encantando por el nilon": "encantado por el nilo",
    "encantado por el nilo": "encantado por el nilo",
    "cronicas de el cairo": "cronicas de el cairo",
    "crónicas de el cairo": "cronicas de el cairo",
    "el rey rut": "el rey tut",
    "el rey tut te invita": "el rey tut",
}

def strip_accents(text: str) -> str:
    return ''.join(
        c for c in unicodedata.normalize('NFKD', text)
        if not unicodedata.combining(c)
    )

def normalize_text(text: str) -> str:
    if text is None:
        return ""
    s = str(text).lower().strip()
    s = strip_accents(s)
    s = s.translate(SPANISH_CHAR_MAP)
    s = s.translate(ARABIC_CHAR_MAP)
    s = s.replace('usd', ' ')
    s = s.replace('$', ' ')
    s = s.replace(',', '')
    s = s.replace('→', ' ')
    s = s.replace('—', ' ')
    s = s.replace('-', ' ')
    s = re.sub(r'\s+', ' ', s)

    for k, v in GENERIC_REPLACEMENTS.items():
        s = s.replace(k, v)

    return s.strip()

def tokenize(text: str) -> List[str]:
    return re.findall(r'[\w]+', normalize_text(text))

def contains_phrase(text: str, phrase: str) -> bool:
    t = normalize_text(text)
    p = normalize_text(phrase)
    return p in t

def token_overlap_ratio(text: str, phrase: str) -> float:
    """
    Soft matching for noisy outputs.
    """
    t_tokens = set(tokenize(text))
    p_tokens = [tok for tok in tokenize(phrase) if len(tok) >= 2]
    if not p_tokens:
        return 0.0
    hits = sum(1 for tok in p_tokens if tok in t_tokens)
    return hits / len(p_tokens)

def phrase_match_score(text: str, phrase: str) -> float:
    """
    Returns:
      1.0 if exact normalized phrase found
      0.7 if strong token overlap
      0.4 if partial overlap
      0.0 otherwise
    """
    if contains_phrase(text, phrase):
        return 1.0
    ratio = token_overlap_ratio(text, phrase)
    if ratio >= 0.8:
        return 0.7
    if ratio >= 0.5:
        return 0.4
    return 0.0

def score_keywords_any(text: str, items: List[str]):
    if not items:
        return None
    scores = [phrase_match_score(text, item) for item in items]
    return max(scores) if scores else 0.0

def score_keywords_all(text: str, items: List[str]):
    if not items:
        return None
    scores = [phrase_match_score(text, item) for item in items]
    return sum(scores) / len(scores) if scores else 0.0

def extract_numbers(text: str) -> List[str]:
    s = normalize_text(text)
    return re.findall(r'(?<!\d)\d+(?:\.\d+)?(?!\d)', s)

def score_numbers(text: str, expected_numbers: List[str]):
    """
    Fairer than old logic:
    - old code gave full credit if ANY one expected number appeared
    - new code gives fractional credit based on how many expected numbers appear
    """
    if not expected_numbers:
        return None

    found = set(extract_numbers(text))
    exp = [str(n).replace(',', '') for n in expected_numbers]

    hits = 0
    for n in exp:
        if n in found:
            hits += 1
        else:
            try:
                if any(abs(float(x) - float(n)) < 1e-9 for x in found if re.match(r'^\d+(\.\d+)?$', x)):
                    hits += 1
            except:
                pass

    return hits / len(exp)

def score_forbidden(text: str, must_not_include: List[str]) -> float:
    """
    Returns penalty in [0,1].
    """
    if not must_not_include:
        return 0.0
    penalties = [phrase_match_score(text, bad) for bad in must_not_include]
    return max(penalties) if penalties else 0.0

def detect_language(text: str) -> str:
    s = str(text)
    if re.search(r'[\u0600-\u06FF]', s):
        return 'ar'
    if any(x in s.lower() for x in ['¿', '¡', ' que ', ' incluye', ' no incluye', ' dia ', ' día ']):
        return 'es'
    return 'en'

def language_score(question: str, answer: str) -> float:
    """
    Soft language compliance score.
    """
    q = question.lower()
    q_lang = detect_language(question)
    a_lang = detect_language(answer)

    if "answer in spanish" in q:
        return 1.0 if a_lang == 'es' else 0.0
    if "answer in arabic" in q:
        return 1.0 if a_lang == 'ar' else 0.0
    if "answer in english" in q:
        return 1.0 if a_lang == 'en' else 0.0

    return 1.0

def category_weights(category: str) -> Dict[str, float]:
    """
    Per-category weighting.
    """
    weights = {
        'kw_any': 0.4,
        'kw_all': 0.5,
        'numbers': 0.4,
        'language': 0.1,
        'forbidden_penalty': 0.7,
    }

    if category == 'pricing':
        weights.update({'kw_any': 0.25, 'numbers': 0.65, 'language': 0.10})
    elif category == 'rules':
        weights.update({'kw_any': 0.25, 'kw_all': 0.65, 'numbers': 0.0, 'language': 0.10})
    elif category == 'traps':
        weights.update({'kw_any': 0.60, 'numbers': 0.0, 'language': 0.10, 'forbidden_penalty': 1.0})
    elif category == 'history':
        weights.update({'kw_any': 0.70, 'numbers': 0.0, 'language': 0.10})
    elif category == 'style':
        weights.update({'kw_any': 0.50, 'numbers': 0.0, 'language': 0.50})

    return weights

def compute_accuracy(ex: Dict[str, Any], answer_text: str) -> Dict[str, Any]:
    category = ex.get('category', '')
    w = category_weights(category)

    kw_any_score = score_keywords_any(answer_text, ex.get('expected_keywords_any', []))
    kw_all_score = score_keywords_all(answer_text, ex.get('expected_keywords_all', []))
    num_score = score_numbers(answer_text, ex.get('expected_numbers', []))
    lang_score = language_score(ex.get('q', ''), answer_text)
    forbidden_pen = score_forbidden(answer_text, ex.get('must_not_include', []))

    weighted_sum = 0.0
    used_weight = 0.0

    if kw_any_score is not None and w.get('kw_any', 0) > 0:
        weighted_sum += kw_any_score * w['kw_any']
        used_weight += w['kw_any']

    if kw_all_score is not None and w.get('kw_all', 0) > 0:
        weighted_sum += kw_all_score * w['kw_all']
        used_weight += w['kw_all']

    if num_score is not None and w.get('numbers', 0) > 0:
        weighted_sum += num_score * w['numbers']
        used_weight += w['numbers']

    if w.get('language', 0) > 0:
        weighted_sum += lang_score * w['language']
        used_weight += w['language']

    base_score = (weighted_sum / used_weight) if used_weight > 0 else 0.0

    acc = max(0.0, base_score - forbidden_pen * w.get('forbidden_penalty', 0.7))
    forbidden_hit = int(forbidden_pen >= 0.7)

    return {
        'accuracy': round(acc, 3),
        'kw_any_score': None if kw_any_score is None else round(kw_any_score, 3),
        'kw_all_score': None if kw_all_score is None else round(kw_all_score, 3),
        'num_score': None if num_score is None else round(num_score, 3),
        'lang_score': round(lang_score, 3),
        'forbidden_score': round(forbidden_pen, 3),
        'forbidden_hit': forbidden_hit,
    }

# ----------------------------
# Main evaluation loop
# ----------------------------
rows = []
for idx, ex in enumerate(EVAL):
    t0 = time.time()
    out = answer(ex['q'])
    dt = time.time() - t0
    out_s = str(out)

    metrics = compute_accuracy(ex, out_s)

    rows.append({
        'index': idx,
        'category': ex.get('category', ''),
        'question': ex['q'][:120],
        'latency': round(dt, 2),
        'accuracy': metrics['accuracy'],
        'kw_any_score': metrics['kw_any_score'],
        'kw_all_score': metrics['kw_all_score'],
        'num_score': metrics['num_score'],
        'lang_score': metrics['lang_score'],
        'forbidden_score': metrics['forbidden_score'],
        'forbidden_hit': metrics['forbidden_hit'],
        'answer_preview': out_s[:300]
    })

df = pd.DataFrame(rows)

print('=' * 80)
print('IMPROVED EVALUATION RESULTS')
print('=' * 80)

cat_summary = df.groupby('category').agg(
    count=('accuracy', 'count'),
    mean_accuracy=('accuracy', 'mean'),
    mean_latency=('latency', 'mean'),
    hallucinations=('forbidden_hit', 'sum')
).round(3)

print(cat_summary)
print()
print(f'OVERALL ACCURACY: {df["accuracy"].mean():.1%}')
print(f'TOTAL QUESTIONS:  {len(df)}')
print(f'AVG LATENCY:      {df["latency"].mean():.2f}s')
print(f'HALLUCINATIONS:   {df["forbidden_hit"].sum()}')
print('=' * 80)

print("\nLOW SCORING ANSWERS (< 0.6):")
low_df = df[df['accuracy'] < 0.6][[
    'index', 'category', 'accuracy', 'question', 'answer_preview',
    'kw_any_score', 'kw_all_score', 'num_score', 'forbidden_score'
]]
print(low_df.to_string(index=False))

df
